In [24]:
import pandas as pd, joblib, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

#data loading & preparation
col_names = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']
df = pd.concat([pd.read_csv("KDDTrain+.txt", names=col_names), pd.read_csv("KDDTest+.txt", names=col_names)])
df = df.drop('difficulty', axis=1)

df['label'] = df['label'].str.strip().str.replace('.', '',regex=False)

attack_map = {'normal': 'Normal', 'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS', 'worm': 'DoS', 'mailbomb': 'DoS', 'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe', 'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L', 'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R'}
df['category'] = df['label'].map(attack_map)
df = df.dropna(subset=['category'])

# preprocessing
X = pd.get_dummies(df.drop(['label','category'], axis=1))
labl = LabelEncoder()
y = labl.fit_transform(df['category'])
class_names = list(labl.classes_)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# SMOTE and training
smote_strategy = {i: max(5000, sum(y_train == i)) for i in range(len(class_names))}
smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print("\nTraining decision Tree")
dt = DecisionTreeClassifier(random_state=42, max_depth=20)
dt.fit(X_res, y_res)

# save artifacts
joblib.dump(sc,'scaler.pkl')
joblib.dump(labl,'label_encoder.pkl')
joblib.dump(dt,'dt_model.pkl')

print("\n Decision Tree saved to dt_model.pkl")


Training decision Tree

 Decision Tree saved to dt_model.pkl


In [25]:
# visualization
plt.figure(figsize=(18, 5))

# class Distribution
plt.subplot(1, 3, 1)
sns.countplot(x='category', hue='category', data=df, palette='viridis', legend=False)
plt.title('Class Distribution')
plt.xticks(rotation=45)

# Confusion matrix
plt.subplot(1, 3, 2)
dt_preds = dt.predict(X_test)
cm = confusion_matrix(y_test, dt_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)

plt.title('Confusion Matrix- Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# feature importance
plt.subplot(1, 3, 3)
importances = pd.Series(dt.feature_importances_, index=X.columns).nlargest(10)
importances.plot(kind='barh', color='teal')
plt.title('Top 10 Feature Importances')

plt.tight_layout()
plt.savefig('nids_visuals.png')
plt.show()

In [29]:
from sklearn.neural_network import MLPClassifier

#ANN: multi layer perceptron
print("\nTraining ANN (MLPClassifier)")
ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    early_stopping=True,
    random_state=42,
    verbose=True
)

ann_model.fit(X_train_resampled,y_train_resampled)
ann_predictions = ann_model.predict(X_test)
ann_accuracy = accuracy_score(y_test,ann_predictions)

#ann Confusion matrix
ann_cm = confusion_matrix(y_test, ann_predictions)
plt.figure(figsize=(7,5))
sns.heatmap(ann_cm,annot=True,fmt='d',cmap='Blues',xticklabels=class_names, yticklabels=class_names)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix_ann.png')
plt.close()

print("Saved img confusion_matrix_ann.png")

joblib.dump(ann_model, 'ann_model.pkl')
print(f"\nANN model saved to ann_model.pkl")


Training ANN (MLPClassifier)
Iteration 1, loss = 0.17719373
Validation score: 0.973730
Iteration 2, loss = 0.07056995
Validation score: 0.980576
Iteration 3, loss = 0.06079765
Validation score: 0.980417
Iteration 4, loss = 0.05302679
Validation score: 0.980338
Iteration 5, loss = 0.04812080
Validation score: 0.983601
Iteration 6, loss = 0.04387219
Validation score: 0.980338
Iteration 7, loss = 0.04161701
Validation score: 0.982328
Iteration 8, loss = 0.03764711
Validation score: 0.987024
Iteration 9, loss = 0.03606525
Validation score: 0.987741
Iteration 10, loss = 0.03336854
Validation score: 0.987582
Iteration 11, loss = 0.03115485
Validation score: 0.988059
Iteration 12, loss = 0.03058079
Validation score: 0.986945
Iteration 13, loss = 0.02906030
Validation score: 0.989413
Iteration 14, loss = 0.02788940
Validation score: 0.987422
Iteration 15, loss = 0.02679972
Validation score: 0.989094
Iteration 16, loss = 0.02702747
Validation score: 0.987024
Iteration 17, loss = 0.02550936
Val

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


Saved img confusion_matrix_ann.png

ANN model saved to ann_model.pkl


In [27]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

print('--- FINAL PROJECT SUMMARY FOR REPORT ---')
print(f'Final Dataset Shape: {X.shape}')
print('\n--- Categorical Feature Mapping ---')
print(df['category'].value_counts())

X_test_df = pd.DataFrame(X_test, columns=X.columns)

# predictions
dt_preds = dt.predict(X_test_df)
ann_preds = ann_model.predict(X_test_df)

print('\n--- Performance Comparison ---')
print(f'Decision Tree Accuracy\n {accuracy_score(y_test, dt_preds)}')
print(f'ANN(MLP) Accuracy\n{accuracy_score(y_test, ann_preds)}')

print('\n--- Decision tree Classification report ---')
print(classification_report(y_test, dt_preds, target_names=class_names))
print('\n--- Ann(MLP) Classifcation report---')
print(classification_report(y_test, ann_preds, target_names=class_names))

print('\n--- Minority Class F1-Scores (U2R & R2L) ---')
dt_rep = classification_report(y_test, dt_preds, target_names=class_names, output_dict=True)
ann_rep = classification_report(y_test, ann_preds, target_names=class_names, output_dict=True)

print(f"Decision Tree\nR2L F1: {dt_rep['R2L']['f1-score']}")
print(f"U2R F1: {dt_rep['U2R']['f1-score']}")
print(f"ANN (MLP)\nR2L F1: {ann_rep['R2L']['f1-score']}")
print(f"U2R F1: {ann_rep['U2R']['f1-score']}")

--- FINAL PROJECT SUMMARY FOR REPORT ---
Final Dataset Shape: (148517, 122)

--- Categorical Feature Mapping ---
category
Normal    77054
DoS       53387
Probe     14077
R2L        3880
U2R         119
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(



--- Performance Comparison ---
Decision Tree Accuracy
 0.9942768650686776
ANN(MLP) Accuracy
0.8466199838405601

--- Decision tree Classification report ---
              precision    recall  f1-score   support

         DoS       1.00      1.00      1.00     10678
      Normal       1.00      0.99      0.99     15411
       Probe       0.99      0.99      0.99      2815
         R2L       0.93      0.93      0.93       776
         U2R       0.58      0.75      0.65        24

    accuracy                           0.99     29704
   macro avg       0.90      0.93      0.91     29704
weighted avg       0.99      0.99      0.99     29704


--- Ann(MLP) Classifcation report---
              precision    recall  f1-score   support

         DoS       0.99      0.76      0.86     10678
      Normal       0.94      0.92      0.93     15411
       Probe       0.59      0.87      0.70      2815
         R2L       0.27      0.66      0.39       776
         U2R       0.05      0.92      0.09  